# 04 — Cross-Vocabulary Translation (CCO to Schema.org)

This notebook demonstrates portal traversal as a cross-vocabulary
translation mechanism. A holon containing employee data modeled in
the Common Core Ontologies (CCO) is connected via a portal to a
holon expecting Schema.org-shaped data. The portal's CONSTRUCT query
translates between the two vocabularies.

**CCO** (Common Core Ontologies, v2.1) uses opaque identifiers under
`https://www.commoncoreontologies.org/`. The mapping below documents
which identifiers correspond to which human-readable concepts:

| CCO Identifier | Label |
|---------------|-------|
| `cco:ont00001262` | Person |
| `cco:ont00001180` | Organization |
| `cco:ont00000003` | Designative Name |
| `cco:ont00002070` | Email Address |
| `cco:ont00000984` | Occupation Role |
| `cco:ont00001765` | has text value |
| `cco:ont00000887` | City |

In [ ]:
from holonic import HolonicDataset

ds = HolonicDataset()

# ── Source holon: employee records in CCO ──

ds.add_holon("urn:holon:hr-records", "HR Employee Records")

ds.add_interior("urn:holon:hr-records", '''
    @prefix cco:    <https://www.commoncoreontologies.org/> .
    @prefix rdfs:   <http://www.w3.org/2000/01/rdf-schema#> .

    # Two employees modeled in CCO vocabulary
    <urn:person:alice> a cco:ont00001262 ;             # cco:Person
        rdfs:label "Alice Chen" ;
        cco:ont00001765 "alice.chen@example.com" .     # cco:has_text_value

    <urn:name:alice> a cco:ont00000003 ;               # cco:DesignativeName
        cco:ont00001765 "Alice Chen" .                 # cco:has_text_value

    <urn:role:alice-eng> a cco:ont00000984 ;           # cco:OccupationRole
        rdfs:label "Senior Engineer" .

    <urn:person:bob> a cco:ont00001262 ;               # cco:Person
        rdfs:label "Bob Martinez" ;
        cco:ont00001765 "bob.martinez@example.com" .   # cco:has_text_value

    <urn:name:bob> a cco:ont00000003 ;                 # cco:DesignativeName
        cco:ont00001765 "Bob Martinez" .               # cco:has_text_value

    <urn:role:bob-mgr> a cco:ont00000984 ;             # cco:OccupationRole
        rdfs:label "Project Manager" .
''')

# ── Target holon: directory expecting Schema.org ──

ds.add_holon("urn:holon:directory", "Company Directory")

print(f"Source holon: {ds.list_holons()[0].iri}")
print(f"Target holon: {ds.list_holons()[1].iri}")

## The translation portal

The portal's CONSTRUCT query maps CCO concepts to Schema.org:

| CCO | Schema.org |
|-----|-----------|
| `cco:ont00001262` (Person) | `schema:Person` |
| `cco:ont00000003` (DesignativeName) with `cco:ont00001765` | `schema:name` |
| `cco:ont00001765` (has text value) on Person | `schema:email` |
| `cco:ont00000984` (OccupationRole) | `schema:jobTitle` |

In [ ]:
ds.add_portal(
    "urn:portal:hr-to-directory",
    source_iri="urn:holon:hr-records",
    target_iri="urn:holon:directory",
    construct_query='''
        PREFIX cco:    <https://www.commoncoreontologies.org/>
        PREFIX schema: <https://schema.org/>
        PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>

        CONSTRUCT {
            ?schemaPerson a schema:Person ;
                schema:name  ?name ;
                schema:email ?email ;
                schema:jobTitle ?jobTitle .
        }
        WHERE {
            GRAPH ?g {
                ?person a cco:ont00001262 ;           # cco:Person
                    cco:ont00001765 ?email .           # has_text_value (email)

                ?nameEntity a cco:ont00000003 ;       # DesignativeName
                    cco:ont00001765 ?name .            # has_text_value

                ?role a cco:ont00000984 ;             # OccupationRole
                    rdfs:label ?jobTitle .

                # Filter: only match names that correspond to this person
                FILTER(CONTAINS(STR(?nameEntity), REPLACE(STR(?person), "urn:person:", "urn:name:")))
                FILTER(CONTAINS(STR(?role), REPLACE(STR(?person), "urn:person:", "urn:role:")))
            }
            BIND(IRI(CONCAT("urn:directory:", ENCODE_FOR_URI(?name))) AS ?schemaPerson)
        }
    ''',
    label="HR Records -> Company Directory (CCO -> Schema.org)",
)

portals = ds.find_portals_from("urn:holon:hr-records")
print(f"Portal: {portals[0].iri}")
print(f"Label:  {portals[0].label}")

## Traverse the portal

Traversal executes the CONSTRUCT against the source holon's interior,
producing Schema.org-shaped triples. The membrane validates the output
against the target's boundary shapes (if any).

In [ ]:
projected, membrane = ds.traverse(
    source_iri="urn:holon:hr-records",
    target_iri="urn:holon:directory",
    validate=False,
)

print(f"Projected {len(projected)} triples")
print(f"Membrane: {membrane.health.value}")
print()
for s, p, o in sorted(projected):
    print(f"  {s.split('/')[-1]:20s}  {p.split('/')[-1]:12s}  {o}")

## What this demonstrates

The source holon speaks CCO. The target holon speaks Schema.org.
The portal's CONSTRUCT query bridges the vocabularies without either
holon knowing about the other's ontology. This is the holonic model's
fundamental integration mechanism: translation happens at the boundary,
not inside the data.

Key observations:

- The CCO interior uses opaque identifiers (`ont00001262` for Person)
  because that's how CCO is published. The CONSTRUCT query knows both
  vocabularies and maps between them.
- The translation is pure SPARQL, not Python. No custom code runs
  during traversal. If the mapping logic changes, you update the
  portal's `constructQuery` triple in the boundary graph.
- Provenance records which portal produced the translation and when.
  The governance chain (who authorized the mapping, which standard
  it conforms to) is covered in notebook `05_governed_boundaries`.

See also:

- `05_governed_boundaries` — adds governance, stewardship, and
  classification to portal contracts
- `10_projection_plugins` — Python transforms for logic that
  CONSTRUCT alone can't express